# 03 - Reading the curves  *(~6 minutes)*

> **Presenter script.** "If you only remember one notebook from today, make it
> this one. Training a model is easy. Knowing whether the thing you just trained
> is any good - and what to change if it isn't - is the actual skill. And it
> mostly comes down to looking at two lines on a chart."

### The one analogy that makes this click

Imagine a student revising for an exam with a book of past papers.

* **Studying** - they learn the underlying ideas. They do well on the past
  papers *and* on the real exam.
* **Memorising the answer key** - they learn "question 3's answer is B". Perfect
  on the past papers. Lost in the real exam.

The **training set** is the past papers. The **validation set** is the real
exam. Everything below is about telling those two students apart.

In [ ]:
# --- boilerplate: make `import minigpt` work no matter where Jupyter started ---
import pathlib
import sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "minigpt").is_dir())
sys.path.insert(0, str(ROOT))

import torch

torch.set_num_threads(4)  # plenty for a model this small; more threads is not faster
print("repo root:", ROOT)

In [ ]:
from minigpt import data
from minigpt import train as T
from minigpt.model import DEFAULT_BATCH_SIZE, MiniGPT, default_config
from minigpt.plots import plot_bars, plot_comparison, plot_curves, use_stream_style

import math
import matplotlib.pyplot as plt

use_stream_style()

tokenizer = data.load_tokenizer()
random_baseline = math.log(tokenizer.vocab_size)

## Presenter switch

Three training runs is more time than a stream has. `scripts/pretrain_all.py`
already ran all three and saved the results.

**Leave this `True` when you are live.** Set it to `False` if you want to prove
the runs are real (about 4 minutes).

In [ ]:
USE_PREBAKED = True

In [ ]:
if USE_PREBAKED:
    underfit = T.load_history("underfit")
    healthy = T.load_history("healthy")
    overfit = T.load_history("overfit")
    print("loaded three pre-baked runs from artifacts/")
else:
    train_data = T.encode_to_tensor(data.load_split("base", "train"), tokenizer)
    val_data = T.encode_to_tensor(data.load_split("base", "val"), tokenizer)
    tiny_data = T.encode_to_tensor(
        (ROOT / "data" / "tiny_train.txt").read_text(encoding="utf-8"), tokenizer
    )

    print("run 1/3: UNDERFIT (learning rate far too small, stopped far too early)")
    T.set_seed(1337)
    m = MiniGPT(default_config(tokenizer.vocab_size))
    underfit = T.train_model(m, train_data, val_data, steps=250, batch_size=DEFAULT_BATCH_SIZE,
                             learning_rate=1e-4, eval_every=25, eval_iters=16,
                             eval_batch_size=32, warmup_steps=10, name="underfit")

    print("\nrun 2/3: HEALTHY")
    T.set_seed(1337)
    m = MiniGPT(default_config(tokenizer.vocab_size))
    healthy = T.train_model(m, train_data, val_data, steps=1200, batch_size=DEFAULT_BATCH_SIZE,
                            learning_rate=3e-3, eval_every=100, eval_iters=16,
                            eval_batch_size=32, name="healthy")

    print("\nrun 3/3: OVERFIT (only 6 documents, no dropout, 1200 steps)")
    T.set_seed(1337)
    m = MiniGPT(default_config(tokenizer.vocab_size, dropout=0.0))
    overfit = T.train_model(m, tiny_data, val_data, steps=1200, batch_size=DEFAULT_BATCH_SIZE,
                            learning_rate=3e-3, weight_decay=0.0, eval_every=75,
                            eval_iters=16, eval_batch_size=32, cosine_decay=False,
                            name="overfit")

## Pathology 1: **underfitting**

> **underfitting** - the model has not learned enough yet. Not because it is
> broken; because it hasn't had the chance.

Look for: **both** lines still high, **both** still clearly falling when the run
ended, and the loss still close to the random-guessing line.

In [ ]:
plot_curves(underfit, title="UNDERFIT - stopped while it was still learning",
            random_baseline=random_baseline, annotate_best=False)
plt.show()

print(f"final train loss : {underfit.train_loss[-1]:.3f}")
print(f"final val loss   : {underfit.val_loss[-1]:.3f}")
print(f"random guessing  : {random_baseline:.3f}   <- we barely moved away from it")
print(f"still falling?   : {underfit.val_loss[-1] < underfit.val_loss[-2]}")

**What to do about it:** the *opposite* of holding back. Train for more steps.
Raise the learning rate. Make the model bigger. Reduce dropout. Underfitting is
the easy problem - you have not spent enough, so spend more.

## Pathology 2: **generalising** (the healthy one)

Look for: both lines falling **together**, and a **small gap** between them at
the end. A small gap means what the model learned from the homework also works
on the exam.

In [ ]:
plot_curves(healthy, title="HEALTHY - both curves fall together",
            random_baseline=random_baseline)
plt.show()

print(f"final train loss : {healthy.train_loss[-1]:.3f}")
print(f"final val loss   : {healthy.val_loss[-1]:.3f}")
print(f"gap (val - train): {healthy.final_gap:+.3f}   <- close to zero is what we want")

**What to do about it:** keep going, and stop when validation loss stops
improving. Nothing is wrong.

## Pathology 3: **overfitting**

> **overfitting** - the model started memorising the training text instead of
> learning the language.

We forced it: 6 documents only, dropout switched off, 1200 steps. That is the
student with 6 past papers and a very good memory.

Look for: train loss diving towards zero while validation loss **flattens and
then turns upward**. The moment the red line bottoms out is the moment the model
stopped learning and started memorising.

In [ ]:
plot_curves(overfit, title="OVERFIT - validation turns back UP while training keeps falling",
            random_baseline=random_baseline)
plt.show()

best_step = overfit.best_val_step
print(f"best validation loss : {overfit.best_val:.3f} at step {best_step}")
print(f"final validation loss: {overfit.val_loss[-1]:.3f}   <- worse than the best")
print(f"final training loss  : {overfit.train_loss[-1]:.3f}   <- much better than the best")
print(f"gap (val - train)    : {overfit.final_gap:+.3f}   <- a big gap is the fingerprint")
print()
print(f"Everything after step {best_step} made the model WORSE at its real job,")
print("while the training loss kept telling us it was getting better.")

**What to do about it:** more data (by far the best fix), stop earlier (**early
stopping** - keep the checkpoint from the best validation step), add dropout or
weight decay, or make the model smaller.

## All three, side by side

This is the picture to burn into your memory.

In [ ]:
plot_comparison([underfit, healthy, overfit],
                labels=["UNDERFIT\nnot done learning",
                        "HEALTHY\ngeneralising",
                        "OVERFIT\nmemorising"],
                title="The three shapes you will see, forever")
plt.show()

In [ ]:
plot_bars(
    ["underfit", "healthy", "overfit"],
    [underfit.final_gap, healthy.final_gap, overfit.final_gap],
    title="The train/validation gap at the end of each run",
    ylabel="validation loss - training loss",
    colors=["#7f7f7f", "#2ca02c", "#d62728"],
)
plt.show()
print("Near zero = the model is honest about what it knows.")
print("Large and positive = it is much better on text it has seen than on text it has not.")

## The cheat sheet

| what you see | what it's called | what it means in plain English | what to do |
|---|---|---|---|
| Both lines high, both still falling | **Underfitting** | It hasn't finished learning | Train longer, raise the learning rate, or use a bigger model |
| Both lines fall together, small gap | **Generalising** | It's learning the actual pattern | Nothing. Keep going until validation stops improving |
| Train falls, validation flattens then rises | **Overfitting** | It's memorising the homework | More data, stop earlier, add dropout/weight decay, or shrink the model |
| Loss jumps around wildly or becomes `nan` | **Diverging** | The steps are too big | Lower the learning rate; add warmup and gradient clipping |
| Loss stuck flat at `ln(vocab_size)` | **Not learning at all** | Something is wired wrong | Check the data, the labels, and that the optimizer is stepping |
| Validation loss *below* training loss | Usually **dropout**, sometimes a **leak** | Dropout is on during training but off during evaluation | Normal if the gap is small. If validation is way better, check for duplicate documents across the split |

## The memorisation test

A chart is an average. Sometimes you want to catch the model in the act.

Take a passage the model **was** trained on, feed it the first 60 characters,
and see how much of the rest it recites **word for word**. A healthy model
continues plausibly. An overfitted model recites.

In [ ]:
tiny_text = (ROOT / "data" / "tiny_train.txt").read_text(encoding="utf-8")
passage = [p for p in tiny_text.split("\n\n") if len(p) > 200][0]

base_model, _, _ = T.load_checkpoint("base")       # the healthy one
overfit_model, _, _ = T.load_checkpoint("overfit")  # the memoriser

for label, m in [("HEALTHY model", base_model), ("OVERFIT model", overfit_model)]:
    result = T.memorization_score(m, tokenizer, passage, prefix_chars=60)
    print("=" * 76)
    print(label)
    print("=" * 76)
    print(f"we gave it   : ...{result['prefix'][-58:]!r}")
    print(f"truth        : {result['truth'][:70]!r}")
    print(f"it produced  : {result['produced'][:70]!r}")
    print(f"verbatim match: {result['matching_chars']} characters "
          f"({result['matching_fraction']:.0%} of the passage)")
    print()

The overfitted model reproduces a far longer stretch of the original text
character for character. It is not writing - it is reciting.

This is also why notebook 01 spent so long on deduplication: **every duplicate
in your corpus is a nudge towards this behaviour.**

## Two habits worth stealing

1. **Plot validation loss every single run.** Training loss on its own is a
   press release. Validation loss is the audit.
2. **Save the checkpoint at the best validation step, not the last one.** They
   are often not the same checkpoint - as run 3 shows very clearly.

**Next:** `04_continued_pretraining.ipynb` - what happens when you take a
finished model and keep training it on something new.